In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
from ensemble import compute_ensemble
import sys
sys.path.append('../')
from aux_func import get_data
from analysis import * 

import matplotlib as mpl
from mosqlient.prediction_optimize import get_df_pars_ls
from harmonize_preds import aux_functions

# Definir a cor das bordas (spines) como cinza
mpl.rcParams['axes.edgecolor'] = 'gray'

# Definir a cor das linhas dos ticks maiores e menores como cinza
mpl.rcParams['xtick.color'] = 'gray'
mpl.rcParams['ytick.color'] = 'gray'
mpl.rcParams['xtick.labelcolor'] = 'black'
mpl.rcParams['ytick.labelcolor'] = 'black'
plt.rcParams['axes.labelsize'] = 16 # Axis labels
plt.rcParams['xtick.labelsize'] = 14  # X-axis tick labels
plt.rcParams['ytick.labelsize'] = 14  # Y-axis tick labels
plt.rcParams['font.size'] = 16  # General font size
FONT = 12

Challenge: 

In [2]:
challenge = 'dengue_city'
col_region = 'adm_2'

Load data: 

In [3]:
df_dengue_state = get_data(challenge)
df_dengue_state.date = pd.to_datetime(df_dengue_state.date)
df_dengue_state.head()

,date,adm_2,casos
0,2010-01-03,4113700,14
1,2010-01-03,2302503,0
2,2010-01-03,4103701,0
3,2010-01-03,3541406,4
4,2010-01-03,3119401,8


Load ranking: 

In [4]:
remove_models = ['3rd_imdc_-unifesp-_-4mosqueteiras-',
                                      '3rd_imdc_fiocruz_zerolags',
                                        '3rd_imdc_rki_rki_zki_ph',
                               '3rd_imdc_rki_rki_zki_ph_lstm_geo',
                                         '3rd_imdc_unesp_recogna',
 '3rd_imdc_universidad_del_valle_grupo_modelamiento_datos_dengue',
 '3rd_imdc_unesp_recogna_daniel']

Load preds: 

In [5]:
df_preds = pd.read_csv(f'../predictions/predictions_all_models_{challenge}.csv.gz', index_col = 'Unnamed: 0')
df_preds.date = pd.to_datetime(df_preds.date)
df_preds = df_preds.loc[~df_preds.model.isin(remove_models)]
#df_preds = df_preds.loc[df_preds.model != model_baseline]


df_preds.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,adm_2,id,validation,wis,model
0,2022-10-09,0.0,0.0,0.0,0.000000,0.263369,0.752038,2.204613,2.548759,3.217477,2931350,13696,1,9.91,3rd_imdc_emap_epidematicos_prophet_fixed
1,2022-10-16,0.0,0.0,0.0,0.000000,0.305167,0.864584,2.260565,2.571846,2.903148,2931350,13696,1,9.91,3rd_imdc_emap_epidematicos_prophet_fixed
2,2022-10-23,0.0,0.0,0.0,0.000000,0.356236,0.934832,2.347639,2.838857,3.055871,2931350,13696,1,9.91,3rd_imdc_emap_epidematicos_prophet_fixed
3,2022-10-30,0.0,0.0,0.0,0.000000,0.329178,0.977098,2.329630,3.129482,3.404810,2931350,13696,1,9.91,3rd_imdc_emap_epidematicos_prophet_fixed
4,2022-11-06,0.0,0.0,0.0,0.019425,0.438958,1.068266,2.553999,3.291094,4.628868,2931350,13696,1,9.91,3rd_imdc_emap_epidematicos_prophet_fixed


In [6]:
df_preds.model.unique().shape

(8,)

Load forecasts:

In [7]:
df_for = pd.read_csv(f'../predictions/forecast_all_models_{challenge}.csv.gz', index_col = 'Unnamed: 0')
df_for.date = pd.to_datetime(df_for.date)
df_for = df_for.loc[~df_for.model.isin(remove_models)]
#df_for = df_for.loc[df_for.model != model_baseline]

#df_for = df_for.loc[df_for[col_region] != 32]

df_for.head()

,date,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95,adm_2,id,validation,model
780,2026-10-11,0.024717,0.041244,0.102585,0.301943,1.311677,5.059768,14.451320,30.575402,44.130657,2931350,14004,Forecast,3rd_imdc_emap_lstm_muni
781,2026-10-18,0.032611,0.059003,0.137782,0.502689,1.868866,5.491871,17.733885,39.238301,60.347400,2931350,14004,Forecast,3rd_imdc_emap_lstm_muni
782,2026-10-25,0.015177,0.032742,0.085030,0.307743,1.121309,4.232050,12.525038,23.402618,39.344678,2931350,14004,Forecast,3rd_imdc_emap_lstm_muni
783,2026-11-01,0.028762,0.044769,0.127912,0.423790,1.430360,5.476913,18.760642,31.867937,49.821311,2931350,14004,Forecast,3rd_imdc_emap_lstm_muni
784,2026-11-08,0.014884,0.033059,0.063121,0.226481,0.814174,2.931963,11.644227,26.774754,41.047947,2931350,14004,Forecast,3rd_imdc_emap_lstm_muni


In [8]:
df_for.model.unique().shape

(8,)

In [9]:
np.setdiff1d(df_for.model.unique(), df_preds.model.unique())

array([], dtype=object)

In [10]:
np.setdiff1d( df_preds.model.unique(), df_for.model.unique())

array([], dtype=object)

In [11]:

df_median_ens = compute_ensemble(df_for, col_region = col_region)
    
df_median_ens.head()

df_median_ens.head()

,date,adm_2,lower_95,lower_90,lower_80,lower_50,pred,upper_50,upper_80,upper_90,upper_95
0,2026-10-11,1200203,0.029684,0.164766,0.565111,2.095487,9.822588,21.393518,33.590177,93.177359,151.683158
1,2026-10-11,1200401,0.542310,1.035545,2.774073,4.458344,10.210685,19.606250,32.775767,47.034943,85.756946
2,2026-10-11,1716109,0.000054,0.000157,0.055880,0.201414,1.347089,3.381098,8.481363,14.518647,22.420662
3,2026-10-11,2302503,0.000082,0.000293,0.000971,0.051871,1.332635,2.102035,6.344745,12.321510,17.223097
4,2026-10-11,2931350,0.011761,0.020622,0.075209,0.492997,3.328960,8.352021,19.518590,35.287701,53.077828


In [12]:
df_median_ens.adm_2.unique()

array([1200203, 1200401, 1716109, 2302503, 2931350, 2933307, 3119401,
       3541406, 3549805, 4103701, 4104808, 4113700, 5102637, 5201405,
       5215231])

In [13]:
df_median_ens.to_csv(f'../predictions/ensemble_weekly_2026_{challenge}.csv.gz', index = False)